# DS675 Mini-Project: Diabetes Health Indicators
## Random Forest - Hyperparameter tunning and feature tunning




In [1]:
# imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, f1_score, recall_score,
                             precision_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')
%matplotlib inline

## 0. Setup — load file.
### This is a binary, balanced version of the dataset

In [2]:
#Load set 
# Load pre-balanced 50/50 dataset
df_full = pd.read_csv('data/diabetes_binary_health_indicators_BRFSS2015.csv')
print('Balanced dataset shape:', df_full.shape)
print()
print('Target distribution (balanced):')
print(df_full['Diabetes_binary'].value_counts())

Balanced dataset shape: (253680, 22)

Target distribution (balanced):
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64


In [3]:
#load file from drive.
from google.colab import drive
import os

# 1. Mount your Google Drive
# You will be prompted to grant permission the first time you run this
drive.mount('/content/drive')

# 2. Define the path (Option A: MyDrive with no space)
# Note: Ensure "Colab Notebooks" matches the exact spelling/capitalization in your Drive
csv_path = "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv"

# 3. Verify if the file exists at that path
if os.path.exists(csv_path):
    print("Success! File found.")
    # 4. Read the CSV
    df_full = pd.read_csv(csv_path)

    # Display the first 5 rows to confirm data loaded correctly
    #print(df.head())
else:
    print("File not found at the specified path.")
    print("Check your folder names for typos or try 'My Drive' with a space.")
#Load set 
# Load pre-balanced 50/50 dataset
df_full = pd.read_csv('data/diabetes_binary_5050split_health_indicators_BRFSS2015.csv')
print('Balanced dataset shape:', df_full.shape)
print()
print('Target distribution (balanced):')
print(df_full['Diabetes_binary'].value_counts())

ModuleNotFoundError: No module named 'google.colab'

##1. Train and Scale

In [4]:
# split, scale
X = df_full.drop('Diabetes_binary', axis=1)
y = df_full['Diabetes_binary']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_std = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_std = pd.DataFrame(X_test_scaled, columns=X.columns)

In [5]:
print('Train:', X_train_std.shape, '| Test:', X_test_std.shape)

Train: (202944, 21) | Test: (50736, 21)


In [6]:
# Balance with SMOTE
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
oversample = SMOTE(random_state=42)
X_train_std, y_train = oversample.fit_resample(X_train_std, y_train)

In [7]:
print(y_train.value_counts(), " ", y_test.value_counts())

Diabetes_binary
0.0    174667
1.0    174667
Name: count, dtype: int64   Diabetes_binary
0.0    43667
1.0     7069
Name: count, dtype: int64


## 2. Hyperparameter tuning for RF

In [8]:
param_grid = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [None, 10],
    'criterion' : ['gini', 'entropy'],
    'random_state':[42, 0]
}

rf = RandomForestClassifier()
grid_RF = GridSearchCV(estimator=rf, param_grid=param_grid, cv=3, scoring='accuracy')
grid_RF.fit(X_train_std, y_train)

,estimator,RandomForestClassifier()
,param_grid,"{'criterion': ['gini', 'entropy'], 'max_depth': [None, 10], 'n_estimators': [100, 150, ...], 'random_state': [42, 0]}"
,scoring,'accuracy'
,n_jobs,None
,refit,True
,cv,3
,verbose,0
,pre_dispatch,'2*n_jobs'
,error_score,nan
,return_train_score,False
,n_estimators,300


## 3. Results for the best RF
#### This is the best result found for RF for our dataset

In [9]:
########################
#Read results
########################
results = pd.DataFrame(grid_RF.cv_results_)

# Filtering to the most important columns for readability
# Filtering to the most important columns for readability
relevant_columns = [
    'param_n_estimators',
    'param_max_depth',
    'param_criterion',
    'param_random_state',
    'mean_test_score',
    'rank_test_score'
]
all_results = results[relevant_columns].sort_values(by='rank_test_score')

print("--- Top 10 Parameter Combinations ---")
print(all_results.head(10))

print(f"\nBest Parameters: {grid_RF.best_params_}")
print(f"Best Cross-Validation Score: {grid_RF.best_score_:.4f}")

# 5. Use the best model to predict
best_RF = grid_RF.best_estimator_
final_predictions = best_RF.predict(X_test_std)

# 6. Evaluate

print(f"Accuracy: {accuracy_score(y_test, final_predictions):.2%}")
classificationreport = classification_report(y_test, final_predictions)
print(classificationreport)

--- Top 10 Parameter Combinations ---
    param_n_estimators param_max_depth param_criterion  param_random_state  \
6                  300            None            gini                  42   
7                  300            None            gini                   0   
4                  200            None            gini                  42   
2                  150            None            gini                  42   
22                 300            None         entropy                  42   
21                 200            None         entropy                   0   
18                 150            None         entropy                  42   
23                 300            None         entropy                   0   
16                 100            None         entropy                  42   
5                  200            None            gini                   0   

    mean_test_score  rank_test_score  
6          0.902976                1  
7          0.902752      

##4. Create datasets with 1 - 21 features.
####ordered by the importance given in SHAP analysis

In [10]:
# Shap chose list of features, on this dataset with SMOTE
sorted_features = ['GenHlth', 'Age', 'BMI', 'HighBP', 'Income', 'HighChol', 'PhysHlth', 'Education', 
                   'Sex', 'MentHlth', 'CholCheck', 'HvyAlcoholConsump', 'HeartDiseaseorAttack', 'DiffWalk', 
                   'Smoker', 'NoDocbcCost', 'Veggies', 'Fruits', 'AnyHealthcare', 'PhysActivity', 'Stroke']
#####################################
# create data frames with k features
#####################################
X_trains = list(range(22))
for i in range(1, 22):
    X_trains[i] = X_train_std[sorted_features[0:i]].copy()

X_tests = list(range(22))
for i in range(1, 22):
    X_tests[i] = X_test_std[sorted_features[0:i]].copy()

##5. Fit the best RF to each of the partial models

In [11]:
from sklearn.base import clone
######################
#feature counting
#Find the number of features needed for high accurracy
######################

#keep scores
accuracy_scores = list(range(22))

# fit the best modals to the partial datasets
for i in range(1, 22):
  # create a clone to make sure no data from last training attaches
  best_RF_p = clone(best_RF)
  best_RF_p.fit(X_trains[i], y_train)
  final_predictions = best_RF_p.predict(X_tests[i])

  #Results
  print("\n"+"*"*50)
  print(f"Number of features: {i}")
  accuracy_scores[i] = accuracy_score(y_test, final_predictions)
  print(f"Accuracy: {accuracy_score(y_test, final_predictions):.4%}")
  classificationreport = classification_report(y_test, final_predictions)
  print('\nClassification_report : ')
  print(classificationreport)
  print("\n"+"*"*50)



**************************************************
Number of features: 1
Accuracy: 61.0750%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.94      0.58      0.72     43667
         1.0       0.23      0.78      0.36      7069

    accuracy                           0.61     50736
   macro avg       0.59      0.68      0.54     50736
weighted avg       0.84      0.61      0.67     50736


**************************************************

**************************************************
Number of features: 2
Accuracy: 85.0146%

Classification_report : 
              precision    recall  f1-score   support

         0.0       0.87      0.96      0.92     43667
         1.0       0.40      0.14      0.21      7069

    accuracy                           0.85     50736
   macro avg       0.64      0.55      0.56     50736
weighted avg       0.81      0.85      0.82     50736


**************************************************

***

##6. How fast does the partial sets converge to the full set?

In [13]:
#Find when we are preety close to the maximum accurecy
full_acc = accuracy_scores[-1]
found80 = found90 = found95 = False
acc_80 = acc_90 = acc_95 = 21

i = 1
while i < 22 and (not found80 or not found90 or not found95):
  if (accuracy_scores[i] / full_acc) > 0.8 and not found80:
    acc_80 = i
    found80 = True
  if (accuracy_scores[i] / full_acc) > 0.9 and not found90:
    acc_90 = i
    found90 = True
  if (accuracy_scores[i] / full_acc) > 0.95 and not found95:
    acc_95 = i
    found95 = True
  i = i+1

print(f"80% Number of features: {acc_80}")
print(f"Accuracy: {accuracy_scores[acc_80] / full_acc:.4%}")

print(f"90% Number of features: {acc_90}")
print(f"Accuracy: {accuracy_scores[acc_90] / full_acc:.4%}")

print(f"95% Number of features: {acc_95}")
print(f"Accuracy: {accuracy_scores[acc_95] / full_acc:.4%}")


80% Number of features: 2
Accuracy: 99.9791%
90% Number of features: 2
Accuracy: 99.9791%
95% Number of features: 2
Accuracy: 99.9791%


## 7. Conclusion

In [14]:
features = sorted_features[0:acc_95]
print(f"KNN has {accuracy_scores[-1]:.2%} accuracy when considering all features." )
print(f"When considering just the features  { {*features} }  the accuracy is {accuracy_scores[acc_95]:.2%}")
print(f"which is {accuracy_scores[acc_95] / full_acc:.2%}  of the whole set accuracy")

KNN has 85.03% accuracy when considering all features.
When considering just the features  {'Age', 'GenHlth'}  the accuracy is 85.01%
which is 99.98%  of the whole set accuracy


In [15]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/RFTuning.ipynb"


This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
--execute
    Execute the notebook prior to export.
    Equivalent to: [--ExecutePr

[NbConvertApp] WARNING | pattern '/content/drive/MyDrive/Colab Notebooks/DS675/mini_project/RFTuning.ipynb' matched no files
